# **Data Quality & Business KPI Monitoring System**

**1. Loading Libraries**

In [ ]:
import pandas as pd
import numpy as np

**2. Reading and loading dataset**

In [ ]:
df = pd.read_csv("/content/sample_data/OnlineRetail.csv", encoding="latin1")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


**3. Data Quality Checks - Missing Values**

In [ ]:
missing_customer = df[df["CustomerID"].isna()]
missing_customer_count = missing_customer.shape[0]

missing_customer_count

135080

**4. Data Quality Checks - Duplicate Invoices**

In [ ]:
duplicate_invoices = df[
    df.duplicated(subset=["InvoiceNo", "StockCode"], keep=False)
]
duplicate_invoice_count = duplicate_invoices.shape[0]

duplicate_invoice_count

20378

5. Data Quality Checks - Invalid Quantities

In [ ]:
is_cancellation = df["InvoiceNo"].astype(str).str.startswith("C")

invalid_quantity = df[(df["Quantity"] <= 0) & (~is_cancellation)]
invalid_quantity_count = invalid_quantity.shape[0]

invalid_quantity_count

1336

**6. Data Quality Checks - Invalid Prices**


In [ ]:
invalid_price = df[df["UnitPrice"] < 0]
invalid_price_count = invalid_price.shape[0]

invalid_price_count

2

**7. Data Quality Checks - Future Dates**

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

future_dates = df[df["InvoiceDate"] > pd.Timestamp.today()]
future_date_count = future_dates.shape[0]

future_date_count

0

**8. Data Quality Summary Report**

In [ ]:
total_records = len(df)
total_records

541909

In [ ]:
dq_summary = pd.DataFrame({
    "Issue Type": [
        "Missing Customer ID",
        "Duplicate Invoice Numbers",
        "Invalid Quantity",
        "Invalid Unit Price",
        "Future Invoice Dates"
    ],
    "Failed Records": [
        missing_customer_count,
        duplicate_invoice_count,
        invalid_quantity_count,
        invalid_price_count,
        future_date_count
    ]
})

dq_summary

,Issue Type,Failed Records
0,Missing Customer ID,135080
1,Duplicate Invoice Numbers,20378
2,Invalid Quantity,1336
3,Invalid Unit Price,2
4,Future Invoice Dates,0


In [ ]:
dq_summary.columns

Index(['Issue Type', 'Failed Records'], dtype='object')

In [ ]:
dq_summary["Failure Percentage"] = round(
    (dq_summary["Failed Records"] / total_records) * 100, 2
)

In [ ]:
dq_summary

,Issue Type,Failed Records,Failure Percentage
0,Missing Customer ID,135080,24.93
1,Duplicate Invoice Numbers,20378,3.76
2,Invalid Quantity,1336,0.25
3,Invalid Unit Price,2,0.00
4,Future Invoice Dates,0,0.00


**9. Data Quality Score Calculation**

In [ ]:
# -----------------------------
# DATA QUALITY SCORE CALCULATION
# -----------------------------

# Define weights for each check (must match order)
weights = [0.3, 0.3, 0.2, 0.1, 0.1]

# Calculate weighted impact
dq_summary["Weighted Impact"] = dq_summary["Failure Percentage"] * weights

# Final data quality score
dq_score = 100 - dq_summary["Weighted Impact"].sum()

dq_score


np.float64(91.343)

In [ ]:
pd.DataFrame({
    "run_date": [pd.Timestamp.today().date()],
    "data_quality_score": [dq_score]
}).to_csv("data_quality_score.csv", index=False)


In [ ]:
total_records = df.shape[0]

dq_summary["Failure Percentage"] = round(
    (dq_summary["Failed Records"] / total_records) * 100, 2
)

dq_summary

,Issue Type,Failed Records,Failure Percentage,Weighted Impact
0,Missing Customer ID,135080,24.93,7.479
1,Duplicate Invoice Numbers,20378,3.76,1.128
2,Invalid Quantity,1336,0.25,0.050
3,Invalid Unit Price,2,0.00,0.000
4,Future Invoice Dates,0,0.00,0.000


In [ ]:
dq_summary.to_csv("python_data_quality_summary.csv", index=False)

**10. Anomaly Detection - Revenue Analysis**

In [ ]:
## Anomaly Detection – Revenue Spikes & Drops
# Ensure InvoiceDate is datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

# Create Revenue column
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# Aggregate daily revenue
daily_revenue = (
    df.groupby(df["InvoiceDate"].dt.date)["Revenue"]
    .sum()
    .reset_index()
)

# Rename columns for clarity
daily_revenue.columns = ["Date", "Daily Revenue"]

# Add transaction volume (row count per day)
daily_transactions = (
    df.groupby(df["InvoiceDate"].dt.date)
    .size()
    .reset_index(name="Transaction Volume")
)
daily_transactions.columns = ["Date", "Transaction Volume"]
daily_revenue = daily_revenue.merge(daily_transactions, on="Date")

daily_revenue.head()

,Date,Daily Revenue,Transaction Volume
0,2010-12-01,58635.56,3108
1,2010-12-02,46207.28,2109
2,2010-12-03,45620.46,2202
3,2010-12-05,31383.95,2725
4,2010-12-06,53860.18,3878


**11. Statistical Threshold Calculation**

In [ ]:
#stat threshold
mean_revenue = daily_revenue["Daily Revenue"].mean()
std_revenue = daily_revenue["Daily Revenue"].std()

upper_threshold = mean_revenue + 2 * std_revenue
lower_threshold = mean_revenue - 2 * std_revenue

mean_revenue, upper_threshold, lower_threshold


(np.float64(31959.82929180328),
 np.float64(66788.35261957317),
 np.float64(-2868.694035966615))

**12. Anomaly Flagging and Alert Log**

In [ ]:
daily_revenue["Anomaly Type"] = daily_revenue["Daily Revenue"].apply(
    lambda x: "High Spike" if x > upper_threshold
    else ("Sharp Drop" if x < lower_threshold else "Normal")
)

# Export the COMPLETE table here, at the end, after everything is added
daily_revenue.to_csv("daily_revenue_full.csv", index=False)

alert_log = daily_revenue.loc[
    daily_revenue["Anomaly Type"] != "Normal"
].copy()
alert_log["Alert Reason"] = "Revenue deviated significantly from historical average"
alert_log
alert_log.to_csv("revenue_anomaly_alert_log.csv", index=False)